# Notebook that compares fish ID assignment with Denis' model

Every recorded EOD in `Data_4fish_2024-02-20T21_16_26.csv` gets a fish ID from the same forward model as
`eod_two_layer_pipeline_clean.ipynb` (quasi-static `solve_instant_field`, two-layer bodies, real masks of that
frame). Two changes from that notebook:

- **Amplitude scales linearly with fish length:** each candidate fish emits `I0 = L`, where `L` is the arc
  length of its midline, instead of unit current.
- **Assignment uses the minimum squared distance:** the recorded electrode-pair vector at the EOD's peak instant
  is scaled to `max|v| = 1`. The simulated vectors of all candidate fish are scaled together, so that the largest
  value over all fish and pairs is 1. This keeps their relative, length-driven amplitudes. Then
  `fish_id = argmin_f min_± ||v_rec ∓ v_sim,f||²`. The `±` is there because the recorded peak can fall on either
  phase of the biphasic EOD.

In [ ]:
import ast, json, gzip, pickle, os, multiprocessing as mp
import numpy as np, pandas as pd

# Reuse the e-field functions (imports, defs, MAT) from the clean pipeline, without running its heavy cells
_src = '\n'.join(''.join(c['source']) for c in json.load(open('eod_two_layer_pipeline_clean.ipynb'))['cells']
                 if c['cell_type'] == 'code')
_tree = ast.parse(_src)
_tree.body = [n for n in _tree.body if isinstance(n, (ast.Import, ast.ImportFrom, ast.FunctionDef))
              or (isinstance(n, ast.Assign) and getattr(n.targets[0], 'id', None) == 'MAT')]
exec(compile(_tree, 'eod_two_layer_pipeline_clean', 'exec'))

In [ ]:
tag      = '16_26'
mgz_file = f'data/video2024-02-20T21_{tag}__223e__59997__masks_packed.mgz'
eod_csv  = f'data/Data_4fish_2024-02-20T21_{tag}.csv'
out_csv  = f'results/fish_id_assignment_{tag}.csv'
pairs    = load_electrode_pairs('data/electrodes_signed_px_coord_2024-02-13.csv')

N, n_poles, delta_h = 100, 100, 0.5   # N=100 is ~5x faster than 200 (needed for ~39k frames)
h = 1.0 / (N - 1)
n_frames_max = None                   # e.g. 200 for a quick test; None = all EOD frames
n_workers = max(os.cpu_count() - 1, 1)

df_eod = pd.read_csv(eod_csv)
print(f"{len(df_eod)} EODs in {df_eod.Frame.nunique()} frames, {len(pairs)} electrode pairs")

In [ ]:
# Recorded ground truth: demeaned pair voltages at each EOD's peak instant, scaled to max|v| = 1
V = np.stack([df_eod[[f'electrode_{pid}_time_{i}' for i in range(1, 26)]].to_numpy(float)
              for pid, _, _ in pairs], axis=1)                       # (n_eod, n_pairs, 25)
V -= V.mean(axis=2, keepdims=True)
t_star = np.abs(V).max(axis=1).argmax(axis=1)
v_rec = V[np.arange(len(V)), :, t_star]
v_rec /= np.abs(v_rec).max(axis=1, keepdims=True)                    # (n_eod, n_pairs)

In [ ]:
def sim_patterns(masks):
    """(n_fish,P,P) masks of one frame -> (n_fish, n_pairs) simulated pair voltages, each fish emitting
       I0 = its midline length; NaN rows for fish missing from the frame."""
    out = np.full((len(masks), len(pairs)), np.nan)
    for f, m in enumerate(masks):
        if m.sum() < 50: continue
        mid = fish_midline_from_mask(m)
        L = np.linalg.norm(np.diff(mid, axis=0), axis=1).sum()          # length scaling (tank units)
        phi, _, _ = solve_instant_field(masks, N, f, L, delta_h, n_poles, **MAT)
        out[f] = [sample_phi_point(phi, pp, h, N) - sample_phi_point(phi, pm, h, N) for _, pp, pm in pairs]
    return out

def assign(v_rec_rows, v_sim):
    """Squared distance of each recorded EOD to every fish (sims jointly scaled to max 1) -> (ids, d2)."""
    s = v_sim / np.nanmax(np.abs(v_sim))
    d2 = np.minimum(((v_rec_rows[:, None] - s) ** 2).sum(-1), ((v_rec_rows[:, None] + s) ** 2).sum(-1))
    d2[:, np.isnan(s).any(1)] = np.inf
    return d2.argmin(1), d2

def iter_frames(mgz_path, wanted):
    """Stream (frame_idx, (n_fish,P,P) masks) for the wanted frames only (same decoding as load_frames_from_mgz)."""
    wanted, last = set(wanted), max(wanted)
    with gzip.open(mgz_path, 'rb') as f:
        pickle.load(f)
        while True:
            try: frame, objs = pickle.load(f)
            except EOFError: return
            if frame > last: return
            if frame in wanted: yield frame, np.stack([_unpack_mask(*objs[o]) for o in sorted(objs)])

In [ ]:
rows_by_frame = df_eod.groupby(df_eod.Frame - 1).indices            # mask frame (0-idx) -> df_eod row positions
todo = sorted(rows_by_frame)[:n_frames_max]
if os.path.exists(out_csv):                                          # resume: skip frames already written
    todo = sorted(set(todo) - set(pd.read_csv(out_csv).Frame - 1))
print(f"{len(todo)} frames to solve")

def write_chunk(frames, sims, header):
    recs = []
    for fr, v_sim in zip(frames, sims):
        r = rows_by_frame[fr]
        ids, d2 = assign(v_rec[r], v_sim)
        recs.append(pd.DataFrame({'eod_row': r, 'Frame': fr + 1, 'Time': df_eod.Time.values[r], 'fish_id': ids,
                                  **{f'd2_fish{f}': d2[:, f] for f in range(d2.shape[1])}}))
    pd.concat(recs).to_csv(out_csv, mode='a', header=header, index=False)

os.makedirs('results', exist_ok=True)
chunk = 8 * n_workers
def flush(buf):
    frames, masks = zip(*buf)
    write_chunk(frames, pool.map(sim_patterns, masks), header=not os.path.exists(out_csv))

with mp.get_context('fork').Pool(n_workers) as pool:
    buf = []
    for k, item in enumerate(iter_frames(mgz_file, todo), 1):
        buf.append(item)
        if len(buf) == chunk:
            flush(buf); buf = []
            if k % (10 * chunk) == 0: print(f"{k}/{len(todo)} frames", flush=True)
    if buf: flush(buf)
print("done")

In [ ]:
res = pd.read_csv(out_csv).sort_values('eod_row')
res.to_csv(out_csv, index=False)
print(f"{len(res)}/{len(df_eod)} EODs assigned -> {out_csv}")
res.fish_id.value_counts().sort_index()